# 12 — `PageMantle-Hard` (`N = 32` Held-Out): Prompt-Matched Baselines, Stage-3 Audit Ablation, Frontier Backbones & Secondary DB-CSE vs Reflexion

**Self-Contained Publication Suite (Resolving Review Issues `M1`, `M3`, `M5`, `M7`, `M8`, `E1`, `E2`, `E4`, `E8`, `Z2`, `Z5`, `Z6`)**:
1. **32 Held-Out Multi-Channel Tasks (`pm_hard_eval_32`, Zero Overlap with `pm_hard_stream_24`)**: Grounded on real `deepnote/pagemantle-company-dataset` records, featuring multi-channel decoy threads, cross-service hidden silence traps, and mixed-intent privacy guards.
2. **Complete Controlled Comparison Across `gpt-4.1-mini`, `gpt-4.1`, and Frontier `gpt-5.6-terra`**:
   - `B0_direct_hard` (Official prompt)
   - **`B0_prompt_matched_direct` (`E1`)** (Official prompt + identical `GENERAL_ENTERPRISE_DISCIPLINE_BLOCK`)
   - `A7_no_skill_routing` (Per-Thread Gate without 8-Skill Contracts)
   - `A4_skeptic_only_gate` (Per-Thread Skeptic-Only Gate + Solo Executor)
   - `DIAL_Collab_no_stage3_audit` (`E8`: Closed-Loop Gate + Skill Routing without Stage-3 Pre-Delivery Auditor)
   - **`DIAL_Collab_full` (Ours: Closed-Loop Deliberative Gate + Skill Routing + Stage-3 Draft–Audit–Refine)**
3. **Secondary Appendix Study (`M5`)**: Fair downstream held-out comparison on `pm_hard_eval_32` after online adaptation on `pm_hard_stream_24`: Static `v0` vs `Reflexion-Append` ($O(T)$ memory) vs `DB-CSE w/o EscrowGate` vs `DB-CSE Full` (bounded $O(1)$ skill cards).
4. **Cross-Family Multi-Judge (`gpt-4.1` + `DeepSeek-V4.1-Flash`) + Paired Wilcoxon/Permutation + Holm–Bonferroni**.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "datasets/pagemantle_hard",
    "caches",
    "runs/pagemantle_hard_v3",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull), Install Dependencies & Hot-Reload ===
REPO_URL = os.environ.get("DIAL_SKILL_REPO_URL", "https://github.com/BoldHu/multiagent-speak.git")
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity, scipy, sklearn  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")

In [ ]:
# === CELL 2: Bootstrap Multi-Provider Secrets & Configs (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import (
    describe_secret_sources,
    load_autodl_api_key,
    load_jev_api_key,
    load_openai_api_key,
)
from dial_skill.config import load_dial_skill_config, load_experiments_config, load_models_config
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role

print(
    "Secret lookup diagnostics:",
    json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT, include_autodl=True), indent=2),
)
OPENAI_API_KEY = load_openai_api_key(drive_root=DRIVE_ROOT, required=True)
JEV_API_KEY = load_jev_api_key(drive_root=DRIVE_ROOT, required=True)
AUTODL_API_KEY = load_autodl_api_key(drive_root=DRIVE_ROOT, required=False)
AUTODL_BASE_URL = os.environ.get("AUTODL_BASE_URL", "https://www.autodl.art/api/v1")

MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print(
    f"OpenAI key: {bool(OPENAI_API_KEY)} | Jev key: {bool(JEV_API_KEY)} | AutoDL key: {bool(AUTODL_API_KEY)}"
)

## Step 1 — Ensure Extracted PageMantle Dataset & Build/Verify `PageMantle-Hard` (56 Tasks)

In [ ]:
# === CELL 3: Ensure PageMantle Records & Build/Load Leak-Free PageMantle-Hard Suite ===
from huggingface_hub import hf_hub_download
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.benchmarks.pagemantle_hard import (
    PageMantleHardSession,
    build_pagemantle_hard_suite,
    extract_feedback_from_hard_episode,
    load_pagemantle_hard_jsonl,
    verify_hard_split_isolation,
)

PM_RAW_DIR = DRIVE_ROOT / "datasets" / "pagemantle"
PM_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Automatically fetch lightweight core PageMantle tarballs from HuggingFace if not yet on Drive
LIGHT_PACKAGES = [
    "data/slack.tar.gz",
    "data/linear.tar.gz",
    "data/calendar.tar.gz",
    "data/search_console.tar.gz",
    "data/analytics.tar.gz",
    "data/stripe.tar.gz",
    "data/notion.tar.gz",
]
for rel_pkg in LIGHT_PACKAGES:
    target_p = PM_RAW_DIR / rel_pkg
    if not target_p.is_file():
        target_p.parent.mkdir(parents=True, exist_ok=True)
        dl = hf_hub_download(repo_id="deepnote/pagemantle-company-dataset", repo_type="dataset", filename=rel_pkg)
        import shutil
        shutil.copyfile(dl, target_p)

pm_env = PageMantleEnv(dataset_dir=PM_RAW_DIR)
pm_env.ensure_extracted()
print("Available extracted PageMantle services:", pm_env.available_services())

HARD_DIR = DRIVE_ROOT / "datasets" / "pagemantle_hard"
EVAL_PATH = HARD_DIR / "pm_hard_eval_32.jsonl"
STREAM_PATH = HARD_DIR / "pm_hard_stream_24.jsonl"

if not (EVAL_PATH.is_file() and STREAM_PATH.is_file()):
    built = build_pagemantle_hard_suite(env=pm_env, seed=2026, output_dir=HARD_DIR)
    eval_tasks = built["pm_hard_eval_32"]
    stream_tasks = built["pm_hard_stream_24"]
else:
    eval_tasks = load_pagemantle_hard_jsonl(EVAL_PATH)
    stream_tasks = load_pagemantle_hard_jsonl(STREAM_PATH)

qa = verify_hard_split_isolation(eval_tasks, stream_tasks, strict_counts=True)
print("PageMantle-Hard QA:", json.dumps(qa, indent=2))

## Step 2 — Run 6-Way Controlled Evaluation Across `gpt-4.1-mini`, `gpt-4.1` & `gpt-5.6-terra`

In [ ]:
# === CELL 4: Execute Controlled Multi-Agent Comparison on pm_hard_eval_32 ===
import pandas as pd
from dial_skill.benchmarks.deepnote_official import (
    DeepnoteOfficialJudge,
    DeepnoteRubricJudgment,
    OfficialDialSkillV2Agent,
    OfficialDirectAgent,
    OfficialEpisodeResult,
    OfficialPromptMatchedDirectAgent,
    summarize_official_8_results,
)
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

HARD_BACKBONES = ["gpt-4.1-mini", "gpt-4.1", "gpt-5.6-terra"]
PRIMARY_JUDGE = "gpt-4.1"
SECONDARY_JUDGE = "DeepSeek-V4.1-Flash" if AUTODL_API_KEY else None

RUN_DIR = DRIVE_ROOT / "runs" / "pagemantle_hard_v3"
RUN_DIR.mkdir(parents=True, exist_ok=True)
EP_PATH = RUN_DIR / "episodes_hard_v3.jsonl"
JG_PATH = RUN_DIR / "judgments_hard_v3.jsonl"

api_cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache_pagemantle_hard.sqlite")
openai_client = OpenAIClient(api_key=OPENAI_API_KEY, models_cfg=MODELS_CFG, cache=api_cache, provider_name="openai")
autodl_client = (
    OpenAIClient(api_key=AUTODL_API_KEY, models_cfg=MODELS_CFG, cache=api_cache, base_url=AUTODL_BASE_URL, provider_name="autodl", timeout_sec=120.0)
    if AUTODL_API_KEY
    else None
)
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=api_cache, openai_client=openai_client, luna_model="gpt-4.1-mini", jev_api_key=JEV_API_KEY)
seed_store = SkillStore.from_json_file()

hard_judge = DeepnoteOfficialJudge(
    openai_client=openai_client,
    judge_model=PRIMARY_JUDGE,
    secondary_openai_client=autodl_client,
    secondary_judge_model=SECONDARY_JUDGE,
    num_passes=2,
    seed=42,
)

def _load_idx(p: Path):
    d = {}
    if p.is_file():
        for ln in p.read_text(encoding="utf-8").splitlines():
            if ln.strip():
                o = json.loads(ln)
                d[(o["method"], o["model_id"], o["task_id"])] = o
    return d

def _append_rec(p: Path, o: dict):
    with p.open("a", encoding="utf-8") as f:
        f.write(json.dumps(o, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

done_eps = _load_idx(EP_PATH)
done_jgs = _load_idx(JG_PATH)

async def eval_hard_method(m_name: str, stored_mid: str, agent):
    print(f"\n--- {m_name} [{stored_mid}] on {len(eval_tasks)} PageMantle-Hard tasks ---")
    for i, task in enumerate(eval_tasks, start=1):
        k = (m_name, stored_mid, task.task_id)
        if k in done_eps:
            ep = OfficialEpisodeResult.model_validate(done_eps[k])
        else:
            async with PageMantleHardSession(hard_task=task, base_env=pm_env) as sess:
                ep = await agent.run_task(sess)
            d = ep.model_dump(mode="json")
            _append_rec(EP_PATH, d)
            done_eps[k] = d

        if k in done_jgs:
            jg = DeepnoteRubricJudgment.model_validate(done_jgs[k])
        else:
            jg = await hard_judge.grade_episode(task=task.to_official_task(), episode=ep)
            jd = jg.model_dump(mode="json")
            _append_rec(JG_PATH, jd)
            done_jgs[k] = jd

        if i % 8 == 0 or i == len(eval_tasks):
            print(f"  [{i:02d}/{len(eval_tasks)}] {task.task_id} -> pred={ep.predicted_action:9s} writes={ep.write_calls_used} score={jg.overall_score:.1f}")

for mid in HARD_BACKBONES:
    try:
        await openai_client.complete_text(model=mid, messages=[{"role": "user", "content": "ping"}], max_tokens=5, use_cache=True)
    except Exception as exc:
        print(f"[SKIP] {mid}: {exc}")
        continue

    cfg_full = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=True, gate_mode="deliberative_collab")
    cfg_no_aud = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=False, gate_mode="deliberative_collab")
    cfg_skep = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=False, gate_mode="skeptic_only")
    cfg_no_skill = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=True, gate_mode="deliberative_collab")
    cfg_no_skill.ablation.disable_skill_routing = True

    suite_specs = [
        ("B0_direct_hard", mid, OfficialDirectAgent(openai_client=openai_client, model=mid, method_name="B0_direct_hard", max_turns=16)),
        ("B0_prompt_matched_direct", mid, OfficialPromptMatchedDirectAgent(openai_client=openai_client, model=mid, method_name="B0_prompt_matched_direct", max_turns=16)),
        ("DIAL_Collab_no_stage3_audit", f"{mid}+{mid}", OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=mid, executor_model=mid, skill_store=seed_store, cfg=cfg_no_aud, method_name="DIAL_Collab_no_stage3_audit", recon_max_calls=8, recon_max_turns=4, stage3_max_turns=16, enable_pre_delivery_audit=False)),
        ("DIAL_Collab_full", f"{mid}+{mid}", OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=mid, executor_model=mid, skill_store=seed_store, cfg=cfg_full, method_name="DIAL_Collab_full", recon_max_calls=8, recon_max_turns=4, stage3_max_turns=16, enable_pre_delivery_audit=True)),
    ]
    if mid == "gpt-4.1":
        suite_specs.extend([
            ("A4_skeptic_only_gate", f"{mid}+{mid}", OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=mid, executor_model=mid, skill_store=seed_store, cfg=cfg_skep, method_name="A4_skeptic_only_gate", recon_max_calls=8, recon_max_turns=4, stage3_max_turns=16, enable_pre_delivery_audit=False)),
            ("A7_no_skill_routing", f"{mid}+{mid}", OfficialDialSkillV2Agent(triage_client=triage_client, openai_client=openai_client, gate_model=mid, executor_model=mid, skill_store=seed_store, cfg=cfg_no_skill, method_name="A7_no_skill_routing", recon_max_calls=8, recon_max_turns=4, stage3_max_turns=16, enable_pre_delivery_audit=True)),
        ])

    for m_name, smid, ag in suite_specs:
        await eval_hard_method(m_name, smid, ag)

## Step 3 — Secondary Appendix Study (`M5`): Online `DB-CSE` vs `Reflexion-Append` on `pm_hard_stream_24` → Held-Out `pm_hard_eval_32`

In [ ]:
# === CELL 5: Fair Downstream Comparison of Bounded DB-CSE vs Unbounded Reflexion on PageMantle-Hard ===
from dial_skill.benchmarks.deepnote_official import GENERAL_ENTERPRISE_DISCIPLINE_BLOCK
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.prompts.templates import REFLEXION_APPEND_PROMPT
from dial_skill.schemas import SharedEvidenceSlate

STREAM_BACKBONE = "gpt-4.1"
REFLEXION_MEM_PATH = RUN_DIR / "reflexion_stream_lessons.json"
DBCSE_STORE_PATH = RUN_DIR / "dbcse_v3_evolved_store.json"

# 1. Build Reflexion-Append lessons across pm_hard_stream_24 and evaluate on held-out pm_hard_eval_32
if REFLEXION_MEM_PATH.is_file():
    reflexion_lessons = json.loads(REFLEXION_MEM_PATH.read_text(encoding="utf-8"))
else:
    reflexion_lessons = []
    for task in stream_tasks:
        mem_block = (
            "PERSISTENT REFLEXION MEMORY FROM PRIOR EPISODES:\n" + "\n".join(f"- {x}" for x in reflexion_lessons)
            if reflexion_lessons
            else ""
        )
        refl_agent = OfficialDirectAgent(
            openai_client=openai_client,
            model=STREAM_BACKBONE,
            method_name="Reflexion_stream",
            extra_system_instructions=f"{GENERAL_ENTERPRISE_DISCIPLINE_BLOCK}\n\n{mem_block}".strip(),
            max_turns=16,
        )
        async with PageMantleHardSession(hard_task=task, base_env=pm_env) as sess:
            ep = await refl_agent.run_task(sess)
        jg = await hard_judge.grade_episode(task=task.to_official_task(), episode=ep)
        fb = extract_feedback_from_hard_episode(ep, task, jg)
        if fb.feedback_type not in ("TP_VALIDATED", "TN_VALIDATED"):
            r_res = await openai_client.complete_text(
                model=STREAM_BACKBONE,
                messages=[
                    {"role": "system", "content": REFLEXION_APPEND_PROMPT},
                    {"role": "user", "content": f"Task Category: {task.category}\nFailure Type: {fb.feedback_type}\nCritique: {fb.critique}"},
                ],
                temperature=0.0,
            )
            if r_res.content.strip():
                reflexion_lessons.append(r_res.content.strip())
    REFLEXION_MEM_PATH.write_text(json.dumps(reflexion_lessons, indent=2), encoding="utf-8")

refl_heldout_agent = OfficialDirectAgent(
    openai_client=openai_client,
    model=STREAM_BACKBONE,
    method_name="Reflexion_append_heldout",
    extra_system_instructions=(
        f"{GENERAL_ENTERPRISE_DISCIPLINE_BLOCK}\n\nPERSISTENT REFLEXION MEMORY:\n"
        + "\n".join(f"- {x}" for x in reflexion_lessons)
    ),
    max_turns=16,
)
await eval_hard_method("Reflexion_append_heldout", STREAM_BACKBONE, refl_heldout_agent)

# 2. Run Bounded DB-CSE on pm_hard_stream_24 and evaluate on held-out pm_hard_eval_32
if DBCSE_STORE_PATH.is_file():
    evolved_store = SkillStore.from_json_file(DBCSE_STORE_PATH)
else:
    evolved_store = SkillStore.from_json_file()
    updater = DBCSEUpdater(
        skill_store=evolved_store,
        openai_client=openai_client,
        critic_model=STREAM_BACKBONE,
        cfg=DIAL_CFG,
        log_path=RUN_DIR / "dbcse_v3_log.jsonl",
    )
    cfg_stream = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=True)
    for task in stream_tasks:
        s_agent = OfficialDialSkillV2Agent(
            triage_client=triage_client,
            openai_client=openai_client,
            gate_model=STREAM_BACKBONE,
            executor_model=STREAM_BACKBONE,
            skill_store=evolved_store,
            cfg=cfg_stream,
            method_name="DIAL_Collab_stream",
            recon_max_calls=8,
            recon_max_turns=4,
            stage3_max_turns=16,
            enable_pre_delivery_audit=True,
            include_primary_slate_in_metadata=True,
        )
        async with PageMantleHardSession(hard_task=task, base_env=pm_env) as sess:
            ep = await s_agent.run_task(sess)
        jg = await hard_judge.grade_episode(task=task.to_official_task(), episode=ep)
        fb = extract_feedback_from_hard_episode(ep, task, jg)
        raw_sl = ep.metadata.get("primary_slate")
        if raw_sl is not None and ep.gate is not None:
            sl_obj = SharedEvidenceSlate.model_validate(raw_sl)
            await updater.process_episode(
                slate=sl_obj,
                gate=ep.gate,
                feedback=fb,
                rescore_fn=lambda sl, st, _a=s_agent, _sk=fb.skill_name: _a.rescore_slate_with_store(slate=sl, candidate_store=st, focus_skill=_sk),
            )
    evolved_store.save(DBCSE_STORE_PATH)

cfg_evo = DialSkillAgent.build_collab_config(DIAL_CFG, enable_pre_delivery_audit=True)
dbcse_heldout_agent = OfficialDialSkillV2Agent(
    triage_client=triage_client,
    openai_client=openai_client,
    gate_model=STREAM_BACKBONE,
    executor_model=STREAM_BACKBONE,
    skill_store=evolved_store,
    cfg=cfg_evo,
    method_name="DIAL_Collab_dbcse_heldout",
    recon_max_calls=8,
    recon_max_turns=4,
    stage3_max_turns=16,
    enable_pre_delivery_audit=True,
)
await eval_hard_method("DIAL_Collab_dbcse_heldout", f"{STREAM_BACKBONE}+{STREAM_BACKBONE}", dbcse_heldout_agent)

## Step 4 — Paired Wilcoxon / Permutation + Holm–Bonferroni Significance & LaTeX Export

In [ ]:
# === CELL 6: Summarize PageMantle-Hard Results, Error Taxonomy & Paired Significance ===
from dial_skill.eval.metrics import holm_bonferroni_correction, paired_nonparametric_test

hard_rows = []
hard_task_vectors = {}

for m_name, smid in sorted({(k[0], k[1]) for k in done_eps}):
    eps_list = [OfficialEpisodeResult.model_validate(done_eps[(m_name, smid, t.task_id)]) for t in eval_tasks if (m_name, smid, t.task_id) in done_eps]
    jg_map = {t.task_id: DeepnoteRubricJudgment.model_validate(done_jgs[(m_name, smid, t.task_id)]) for t in eval_tasks if (m_name, smid, t.task_id) in done_jgs}
    if len(eps_list) < len(eval_tasks) or len(jg_map) < len(eval_tasks):
        continue
    s = summarize_official_8_results(eps_list, jg_map)
    hard_task_vectors[(m_name, smid)] = [jg_map[t.task_id].overall_score for t in eval_tasks]
    hard_rows.append({
        "method": m_name,
        "backbone": smid.split("+")[0],
        **s,
    })

display(pd.DataFrame(hard_rows))

hard_tests = {}
hard_raw_p = {}
for mid in HARD_BACKBONES:
    ours_key = ("DIAL_Collab_full", f"{mid}+{mid}")
    if ours_key not in hard_task_vectors:
        continue
    for b_name, b_mid in [
        ("B0_direct_hard", mid),
        ("B0_prompt_matched_direct", mid),
        ("DIAL_Collab_no_stage3_audit", f"{mid}+{mid}"),
        ("A4_skeptic_only_gate", f"{mid}+{mid}"),
        ("A7_no_skill_routing", f"{mid}+{mid}"),
    ]:
        if (b_name, b_mid) in hard_task_vectors:
            t_key = f"{mid}:DIAL_Collab_full_vs_{b_name}"
            res = paired_nonparametric_test(hard_task_vectors[ours_key], hard_task_vectors[(b_name, b_mid)])
            hard_tests[t_key] = res
            hard_raw_p[t_key] = res["permutation_p_value"]

holm_hard = holm_bonferroni_correction(hard_raw_p)
for k, v in holm_hard.items():
    hard_tests[k].update(v)

display(pd.DataFrame.from_dict(hard_tests, orient="index"))
(RUN_DIR / "summary_pagemantle_hard_v3.json").write_text(
    json.dumps({"rows": hard_rows, "paired_tests": hard_tests}, indent=2), encoding="utf-8"
)
print("Saved PageMantle-Hard v3 summary & statistical significance report!")